# ML-10 · Content Action Playbook

**Lane:** Core Lane 2 — Content Refresh / Content Opportunity Scoring  
**Built on:** Gradient Boosting model (ML-08), validated in ML-09  
**Model P@50:** 0.772 (GroupKFold by client_id)  
**Purpose:** Turn model scores into a ranked, human-reviewable content action queue with reason codes, archetype→action rules, and clear no-go boundaries.

> This playbook is **analytical and non-production**. No action should be automated without human review.

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import sys, json
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

ROOT  = Path('../../')
FEAT  = ROOT / 'data/processed/refresh_feature_vector.csv'
OUT   = ROOT / 'work/outputs'
FIGS  = ROOT / 'work/figures'
OUT.mkdir(parents=True, exist_ok=True)
FIGS.mkdir(parents=True, exist_ok=True)

features = pd.read_csv(FEAT)
print(f'Feature frame: {len(features):,} rows x {len(features.columns)} cols')

---
## 1) Ranked Actions + Reason Codes

In [ ]:
# ── Re-train Gradient Boosting on full dataset for final scoring ──────────────
FEATURE_COLS = [
    'log_impressions_90d', 'log_clicks_90d', 'log_sessions_90d',
    'log_ai_sessions_90d', 'ctr', 'avg_position', 'days_since_last_update',
    'content_age_days', 'search_volume', 'engagement_rate', 'scroll_rate',
    'impressions_last_30d', 'impressions_prev_30d', 'has_clicks',
    'has_ai_sessions', 'measurable_opportunity', 'competition', 'cpc'
]
LABEL_COL = 'is_declining_label'

df = features.dropna(subset=FEATURE_COLS + [LABEL_COL]).copy()
X  = df[FEATURE_COLS]; y = df[LABEL_COL]

pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', GradientBoostingClassifier(n_estimators=200, max_depth=4,
                                       learning_rate=0.05, subsample=0.8,
                                       random_state=42))
])
pipe.fit(X, y)
df['decline_score'] = pipe.predict_proba(X)[:, 1]
print(f'Scored {len(df):,} pages. Score range: {df["decline_score"].min():.3f} - {df["decline_score"].max():.3f}')

In [ ]:
# ── Reason code assignment ────────────────────────────────────────────────────
PALETTE = {
    'FULL_REWRITE':        '#ef4444',
    'CONTENT_UPDATE':      '#f97316',
    'META_TITLE_REFRESH':  '#eab308',
    'UX_IMPROVEMENT':      '#3b82f6',
    'REVIEW_SCHEDULE':     '#8b5cf6',
    'MONITOR':             '#6b7280',
}

def assign_reason_codes(row):
    codes = []
    if row['days_since_last_update'] > 180:           codes.append('STALE_CONTENT')
    if row['avg_position'] <= 10 and row['ctr'] < 0.02: codes.append('PAGE1_LOW_CTR')
    if row['impressions_prev_30d'] > 0:
        if row['impressions_last_30d'] / (row['impressions_prev_30d'] + 1) < 0.7:
            codes.append('IMPRESSION_DROP')
    if row['engagement_rate'] < 0.05:                 codes.append('LOW_ENGAGEMENT')
    if row['search_volume'] > 100 and row['decline_score'] > 0.6:
                                                      codes.append('HIGH_VOLUME_AT_RISK')
    if row['content_age_days'] < 60:                  codes.append('NEW_CONTENT')
    return '|'.join(codes) if codes else 'GENERAL_DECAY'

def assign_action(row):
    codes = row['reason_codes'].split('|')
    score = row['decline_score']
    if 'NEW_CONTENT' in codes:                               return 'MONITOR'
    if 'HIGH_VOLUME_AT_RISK' in codes and score > 0.75:     return 'FULL_REWRITE'
    if 'PAGE1_LOW_CTR' in codes:                            return 'META_TITLE_REFRESH'
    if 'STALE_CONTENT' in codes and 'IMPRESSION_DROP' in codes: return 'CONTENT_UPDATE'
    if 'IMPRESSION_DROP' in codes and score > 0.6:          return 'CONTENT_UPDATE'
    if 'LOW_ENGAGEMENT' in codes:                           return 'UX_IMPROVEMENT'
    if score > 0.65:                                        return 'REVIEW_SCHEDULE'
    return 'MONITOR'

def assign_priority(row):
    if row['recommended_action'] == 'FULL_REWRITE' and row['decline_score'] > 0.75: return 'P1_URGENT'
    if row['recommended_action'] in ('CONTENT_UPDATE','META_TITLE_REFRESH') and row['decline_score'] > 0.6: return 'P2_HIGH'
    if row['recommended_action'] in ('UX_IMPROVEMENT','REVIEW_SCHEDULE'): return 'P3_MEDIUM'
    return 'P4_WATCH'

df['reason_codes']        = df.apply(assign_reason_codes, axis=1)
df['recommended_action']  = df.apply(assign_action, axis=1)
df['priority']            = df.apply(assign_priority, axis=1)

QUEUE_COLS = ['content_id','client_id','decline_score','priority','recommended_action',
              'reason_codes','days_since_last_update','avg_position','ctr',
              'impressions_90d','search_volume','engagement_rate','content_age_days','is_declining_label']
queue = df[QUEUE_COLS].sort_values('decline_score', ascending=False).reset_index(drop=True)
queue['rank'] = queue.index + 1

print('=== ARCHETYPE -> ACTION MAPPING ===')
print(queue['recommended_action'].value_counts().to_string())
print()
print('=== PRIORITY BREAKDOWN ===')
print(queue['priority'].value_counts().to_string())
print()
print('=== TOP-10 QUEUE ===')
print(queue[['rank','content_id','decline_score','priority','recommended_action','reason_codes']].head(10).to_string(index=False))

In [ ]:
# ── Reason code table (top-500) ───────────────────────────────────────────────
reason_flat = []
for codes in queue.head(500)['reason_codes']:
    reason_flat.extend(codes.split('|'))
print('Top reason codes in first 500 pages:')
print(pd.Series(reason_flat).value_counts().head(8).to_string())

### Reason Code Reference

| Code | Signal | Threshold | Interpretation |
|---|---|---|---|
| `STALE_CONTENT` | `days_since_last_update` | > 180 days | Page hasn't been updated in 6+ months |
| `PAGE1_LOW_CTR` | `avg_position <= 10` AND `ctr < 0.02` | — | Ranking well but title/description failing to earn clicks |
| `IMPRESSION_DROP` | `impressions_last_30d / impressions_prev_30d` | < 0.70 | Impressions fell more than 30% month-over-month |
| `LOW_ENGAGEMENT` | `engagement_rate` | < 0.05 | Users arriving but not engaging — content or UX problem |
| `HIGH_VOLUME_AT_RISK` | `search_volume > 100` AND `decline_score > 0.6` | — | High-value keyword with strong decline signal |
| `NEW_CONTENT` | `content_age_days` | < 60 days | Too new to diagnose — needs more data |
| `GENERAL_DECAY` | — | fallback | Score elevated but no single dominant signal |

---
## 2) Intended Use and Limits

### Intended Use
This playbook is designed to help **content editors and SEO managers** prioritise which pages to review for refresh. The `decline_score` is a **probability estimate** (0–1) of whether a page is currently in decline. The ranked queue surfaces the pages where intervention is most likely to recover performance.

The recommended workflow is:
1. Pull the top-100 from the queue each month
2. An editor reviews each page's reason codes and confirms the action
3. A content writer executes the action (refresh, rewrite, meta update)
4. Outcome is tracked 30 days post-refresh

### Limits

| Limit | Why it matters |
|---|---|
| **Client-size skew** | 32 clients, unequal sizes. Large clients dominate training. Scores for small, new clients are less reliable. |
| **Single snapshot** | The model is trained on one month's data. Scores are a snapshot, not a time series. |
| **Grouped CV, not temporal** | GroupKFold by client prevents leakage but doesn't prove the model holds over time. |
| **P@50 = 0.772** | 77% of the top-50 flagged pages are genuinely declining. 23% are false positives — human review catches these. |
| **Observational, not causal** | A high decline score does not prove the page's content caused the decline. External factors (algorithm updates, competitor changes) may be the true cause. |
| **No A/B validation** | We have not measured lift from refreshed vs. unrefreshed pages. The model predicts decline, not refresh ROI. |

---
## 3) Human Review + The No-Go List

### Human Review Rules

Every page in P1/P2 requires **editor sign-off** before action:

1. **Confirm the reason code makes sense** — a `PAGE1_LOW_CTR` flag should be verifiable in Search Console directly. If the editor can't see it, the flag is wrong.
2. **Check for external cause** — if impressions dropped but a major Google algorithm update ran in the same window, the drop may not be content-related.
3. **Check brand sensitivity** — some pages are intentionally minimal (legal, privacy). Never flag these for FULL_REWRITE based on score alone.
4. **New content (<60 days)** — always MONITOR. No refreshes in the first 60 days.
5. **Client approval** — for FULL_REWRITE, client sign-off is required before any work begins.

### ❌ The No-Go List — What Should NOT Be Automated

| Action | Why NOT to automate |
|---|---|
| **FULL_REWRITE** | Replaces substantial content. Irreversible without a rollback process. Requires editorial and client approval. |
| **META_TITLE_REFRESH** | Title changes affect ranking signal. Mass-automated title changes can tank a client's entire site overnight. |
| **Any action on pages with `avg_position <= 3`** | Top-3 positions are fragile. Refreshing content that's ranking #1–3 can cause immediate ranking loss. |
| **Bulk actions on a single client** | If 40 pages from one client appear in the top-50, do NOT action all 40 at once. Stagger over 4–6 weeks. |
| **Actions during algorithm update windows** | Google algorithm updates create false signals. No actions in the 4 weeks following a confirmed core update. |

---
## 4) Monitoring / Retrain Triggers

### Monitoring
The model should be monitored monthly. Track these signals:

| Signal | Check | Alert threshold |
|---|---|---|
| **Score drift** | Mean decline_score vs. prior month | Change > ±0.05 |
| **P@50 on new data** | Apply model to new month, check label agreement | Drop below 0.65 |
| **Feature drift** | Mean `days_since_last_update`, `ctr`, `avg_position` | Any feature shifts > 2 std devs |
| **Action outcome** | 30-day post-refresh impression change | If refreshed pages don't recover, model is over-flagging |

### Retrain Triggers

**Retrain immediately if:**
- P@50 drops below 0.65 on two consecutive months
- A Google core algorithm update is confirmed (label definitions shift)
- The client mix changes significantly (>5 new large clients added)
- Feature distributions shift more than 2 standard deviations from training baseline

**Retrain on schedule:**
- Every 3 months, re-train on the latest 6 months of data
- Always re-validate with GroupKFold before deploying new model

### Cost / Value Thinking

| Action | Estimated cost | Expected value |
|---|---|---|
| FULL_REWRITE | 4–8 writer hours | Recovery of high-volume keyword = high value |
| CONTENT_UPDATE | 1–3 writer hours | Targeted refresh of stale signals |
| META_TITLE_REFRESH | 0.5 hours | Quick win on CTR for page-1 positions |
| UX_IMPROVEMENT | Dev time varies | Engagement uplift, hard to attribute directly |
| MONITOR | 0 | Avoids wasted effort on pages not yet diagnosable |

**Rule of thumb:** Only FULL_REWRITE pages where `search_volume > 50`. For low-volume pages, CONTENT_UPDATE or META_TITLE_REFRESH are the highest ROI actions.

---
## 5) Exports for the Paper

In [ ]:
# ── Export ranked queue ───────────────────────────────────────────────────────
queue_path = OUT / 'ml10_ranked_queue.csv'
queue.to_csv(queue_path, index=False)
print(f'Ranked queue exported: {len(queue):,} rows -> {queue_path}')

top50 = queue.head(50)
top50_precision = top50['is_declining_label'].mean()
print(f'Top-50 precision (ground truth): {top50_precision:.3f}')

In [ ]:
# ── Figure 1: Action mix ──────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5), facecolor='#0d1117')
for ax in axes:
    ax.set_facecolor('#161b22'); ax.tick_params(colors='#e6edf3')
    ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
    for s in ['bottom','left']: ax.spines[s].set_color('#30363d')

action_counts = queue['recommended_action'].value_counts()
colors = [PALETTE.get(a,'#6b7280') for a in action_counts.index]
axes[0].pie(action_counts.values, labels=action_counts.index, colors=colors,
            autopct='%1.1f%%', startangle=140,
            textprops={'color':'#e6edf3','fontsize':9})
axes[0].set_title('Action Mix — Full Queue', color='#e6edf3', fontsize=12, pad=15)

action_order = ['FULL_REWRITE','CONTENT_UPDATE','META_TITLE_REFRESH','UX_IMPROVEMENT','REVIEW_SCHEDULE','MONITOR']
means = queue.head(500).groupby('recommended_action')['decline_score'].mean().reindex(action_order).dropna()
bars  = axes[1].bar(range(len(means)), means.values,
                    color=[PALETTE.get(a,'#6b7280') for a in means.index], edgecolor='none', width=0.6)
axes[1].set_xticks(range(len(means)))
axes[1].set_xticklabels([a.replace('_','\n') for a in means.index], fontsize=8, color='#e6edf3')
axes[1].set_ylabel('Mean Decline Score', color='#8b949e'); axes[1].set_ylim(0,1)
axes[1].set_title('Mean Score by Action (Top-500)', color='#e6edf3', fontsize=12)
for bar, val in zip(bars, means.values):
    axes[1].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.02,
                 f'{val:.2f}', ha='center', va='bottom', color='#e6edf3', fontsize=9)
plt.tight_layout()
plt.savefig(FIGS/'ml10_action_mix.png', dpi=150, bbox_inches='tight', facecolor='#0d1117')
plt.show(); print('Figure 1 saved: work/figures/ml10_action_mix.png')

In [ ]:
# ── Figure 2: Score distribution ──────────────────────────────────────────────
fig2, ax2 = plt.subplots(figsize=(10, 5), facecolor='#0d1117')
ax2.set_facecolor('#161b22'); ax2.tick_params(colors='#e6edf3')
for s in ax2.spines.values(): s.set_color('#30363d')
ax2.spines['top'].set_visible(False); ax2.spines['right'].set_visible(False)
ax2.hist(queue[queue['is_declining_label']==0]['decline_score'], bins=50,
         color='#3b82f6', alpha=0.6, label='Not Declining', edgecolor='none')
ax2.hist(queue[queue['is_declining_label']==1]['decline_score'], bins=50,
         color='#ef4444', alpha=0.6, label='Declining', edgecolor='none')
ax2.axvline(0.6, color='#f59e0b', linestyle='--', linewidth=1.5, label='Action Threshold (0.6)')
ax2.set_xlabel('Decline Probability Score', color='#8b949e')
ax2.set_ylabel('Page Count', color='#8b949e')
ax2.set_title('Score Distribution by True Label', color='#e6edf3', fontsize=13)
ax2.legend(framealpha=0, labelcolor='#e6edf3')
plt.tight_layout()
plt.savefig(FIGS/'ml10_score_distribution.png', dpi=150, bbox_inches='tight', facecolor='#0d1117')
plt.show(); print('Figure 2 saved: work/figures/ml10_score_distribution.png')

In [ ]:
# ── Figure 3: Top reason codes ────────────────────────────────────────────────
reason_flat = []
for codes in queue.head(500)['reason_codes']: reason_flat.extend(codes.split('|'))
reason_series = pd.Series(reason_flat).value_counts().head(8)

fig3, ax3 = plt.subplots(figsize=(10,5), facecolor='#0d1117')
ax3.set_facecolor('#161b22'); ax3.tick_params(colors='#e6edf3')
for s in ax3.spines.values(): s.set_color('#30363d')
ax3.spines['top'].set_visible(False); ax3.spines['right'].set_visible(False)
ax3.barh(range(len(reason_series)), reason_series.values[::-1], color='#8b5cf6', edgecolor='none')
ax3.set_yticks(range(len(reason_series)))
ax3.set_yticklabels(reason_series.index[::-1], color='#e6edf3', fontsize=10)
ax3.set_xlabel('Count in Top-500', color='#8b949e')
ax3.set_title('Top Reason Codes (Top-500 queue)', color='#e6edf3', fontsize=13)
plt.tight_layout()
plt.savefig(FIGS/'ml10_top_reason_codes.png', dpi=150, bbox_inches='tight', facecolor='#0d1117')
plt.show(); print('Figure 3 saved: work/figures/ml10_top_reason_codes.png')

In [ ]:
# ── Metrics JSON ──────────────────────────────────────────────────────────────
metrics = {
    'total_pages_scored':    len(queue),
    'action_threshold':      0.6,
    'p1_urgent_count':       int((queue['priority']=='P1_URGENT').sum()),
    'p2_high_count':         int((queue['priority']=='P2_HIGH').sum()),
    'top50_precision':       round(float(top50_precision), 4),
    'action_mix_full_queue': queue['recommended_action'].value_counts().to_dict(),
    'mean_score_declining':  round(float(queue[queue['is_declining_label']==1]['decline_score'].mean()), 4),
    'mean_score_not_declining': round(float(queue[queue['is_declining_label']==0]['decline_score'].mean()), 4),
    'model_used':            'Gradient Boosting (n_estimators=200, max_depth=4)',
    'validation_p50':        0.772,
    'figures':               ['ml10_action_mix.png','ml10_score_distribution.png','ml10_top_reason_codes.png'],
}
with open(OUT/'ml10_playbook_metrics.json','w') as f: json.dump(metrics, f, indent=2)
print('Metrics saved -> work/outputs/ml10_playbook_metrics.json')
print(json.dumps({k:v for k,v in metrics.items() if k!='action_mix_full_queue'}, indent=2))

---
## 6) Self-Check

| Requirement | Status |
|---|---|
| Intended use explained | ✅ |
| Ranked actions with decline scores | ✅ |
| Reason codes defined and assigned | ✅ |
| Archetype → action mapping | ✅ |
| Limits of the model stated | ✅ |
| Human review rules defined | ✅ |
| No-go list (what NOT to automate) | ✅ |
| Cost / value thinking per action | ✅ |
| Monitoring signals defined | ✅ |
| Retrain triggers defined | ✅ |
| Queue exported to `work/outputs/` | ✅ |
| Figures committed to `work/figures/` | ✅ |
| Metrics JSON committed | ✅ |
| Playbook is non-production / analytical | ✅ |